# Octagon Odds: GPU training run

Runs the full pipeline on a free Colab GPU: **cuDF** for feature engineering, **XGBoost (CUDA)** for the win model, **CuPy** for the Monte Carlo simulator.

**First:** Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## Get the code
Push the repo to GitHub and clone it (edit the URL), or upload `mma-predict.zip` with the file browser on the left and unzip it.

In [ ]:
import os
REPO = 'https://github.com/arjunpaipdx-cell/mma-predict.git'  # <- edit
if os.path.exists('/content/mma-predict.zip'):
    !unzip -qo /content/mma-predict.zip -d /content
elif not os.path.exists('/content/mma-predict'):
    !git clone -q $REPO /content/mma-predict
%cd /content/mma-predict

In [ ]:
# Colab GPU runtimes already include cuDF, CuPy and CUDA XGBoost
!pip -q install altair
from mmapred import backend as B
print(B.describe())
assert B.GPU, 'No GPU found: switch the runtime to T4 GPU'

## 1. Train (GPU hyperparameter sweep)
Chronological split: train before 2023, validate on 2023, test on 2024 onward.

In [ ]:
!python -m mmapred.train --sweep 40

## 2. Check the simulator on held-out fights
All ~1,450 test fights × 100k simulations in one batched GPU run.

In [ ]:
!python -m mmapred.evaluate --sims 100000

## 3. CPU vs GPU benchmarks
These are the numbers for your README and resume. Report what you get, even where the GPU doesn't win (small tables often don't).

In [ ]:
!python benchmarks/run_all.py
from IPython.display import Image
Image('artifacts/benchmarks.png')

## 4. Try a matchup

In [ ]:
from mmapred.predict import Predictor
import json
pr = Predictor()
out, A, B_ = pr.predict('Islam Makhachev', 'Ilia Topuria', rounds=5, n_sims=1_000_000)
print(json.dumps({k: out[k] for k in ('p_a','p_b','by_method','p_finish')}, indent=2))

## 5. Download the trained model
Put the `artifacts/` folder into your local repo, then run `streamlit run app.py` on your laptop (CPU is fine for the app).

In [ ]:
!zip -qr artifacts.zip artifacts
from google.colab import files
files.download('artifacts.zip')